In [ ]:
# @title 1. Setup and Configuration
# --- IMPORTANT ---
# Change the value of `zip_file_path` to the actual path of your uploaded zip file.

# If you are using Google Colab, you can upload your file to the session storage
# by clicking the "Files" icon on the left sidebar and then "Upload to session storage".
# The path will typically be "/content/your_file_name.zip".

zip_file_path = "/content/images.zip"  # <-- CHANGE THIS TO YOUR FILE'S PATH

# --- Library Imports ---
import os
import zipfile
from PIL import Image
import io
from IPython.display import display, FileLink
import multiprocessing
from tqdm.notebook import tqdm  # Use notebook-friendly tqdm

print("Configuration and libraries are ready.")
# Determine the number of CPU cores to use for processing
cpu_cores = multiprocessing.cpu_count()
print(f"Using {cpu_cores} CPU cores for processing.")
print(f"Will attempt to process file at: {zip_file_path}")

In [ ]:
# @title 2. Worker Function for Multiprocessing
# This function defines the work that each individual process will do.
# It takes the filename and byte content of one image, performs the cropping,
# and returns the results as a dictionary.

def process_single_image_worker(image_data):
    """
    Worker function to crop a single image.
    Args:
        image_data (tuple): A tuple containing (filename, image_bytes).
    Returns:
        dict: A dictionary with the original filename and the byte data of the cropped images.
              Returns None if processing fails.
    """
    filename, image_bytes = image_data
    try:
        # Open image from its byte content
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        width, height = img.size

        # --- Crop Right Half ---
        right_half_coords = (width / 2, 0, width, height)
        cropped_right = img.crop(right_half_coords)
        right_buffer = io.BytesIO()
        cropped_right.save(right_buffer, format='JPEG')
        right_bytes = right_buffer.getvalue()

        # --- Crop Top Right Quadrant ---
        top_right_coords = (width / 2, 0, width, height / 2)
        cropped_top_right = img.crop(top_right_coords)
        top_right_buffer = io.BytesIO()
        cropped_top_right.save(top_right_buffer, format='JPEG')
        top_right_bytes = top_right_buffer.getvalue()

        return {
            'filename': os.path.basename(filename),
            'right_half_bytes': right_bytes,
            'top_right_bytes': top_right_bytes
        }
    except Exception as e:
        # Log error for the specific file and return None
        print(f"Could not process file {filename}: {e}")
        return None

In [ ]:
# @title 3. Main Processing Logic with Multiprocessing
# This cell runs the main logic. It reads the zip, distributes the work to
# multiple processes, and then collects the results to create the final zip files.

def process_images_in_parallel(path_to_zip):
    if not os.path.exists(path_to_zip):
        print(f"🚨 ERROR: File not found at '{path_to_zip}'")
        print("Please make sure the path in the first cell is correct and you have uploaded the file.")
        return

    print(f"Processing images from: {path_to_zip}")

    try:
        # --- STAGE 1: Read all image data into memory ---
        # This is done by the main process to avoid file handle conflicts.
        image_data_to_process = []
        with zipfile.ZipFile(path_to_zip, 'r') as original_zip:
            all_files = original_zip.infolist()
            print(f"Reading {len(all_files)} files from zip archive...")
            for item in tqdm(all_files, desc="Reading files"):
                if not item.is_dir() and item.filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')):
                    # Read file bytes and pair with filename
                    image_data_to_process.append((item.filename, original_zip.read(item)))

        if not image_data_to_process:
            print("⚠️ Warning: No compatible image files were found in the zip archive.")
            return

        print(f"\nFound {len(image_data_to_process)} images to process.")

        # --- STAGE 2: Process images in parallel ---
        # A pool of processes is created to run the `process_single_image_worker` function.
        # `imap_unordered` is used for efficiency, processing items as they become available.
        results = []
        with multiprocessing.Pool(processes=cpu_cores) as pool:
            # tqdm shows the progress of the parallel processing
            with tqdm(total=len(image_data_to_process), desc="Cropping images") as pbar:
                for result in pool.imap_unordered(process_single_image_worker, image_data_to_process):
                    if result:  # Only add successful results
                        results.append(result)
                    pbar.update(1)

        # --- STAGE 3: Write results to new zip files ---
        # This is done by the main process to ensure safe writing.
        print("\nWriting processed images to new zip files...")
        right_half_zip_name = 'cropped_right_half.zip'
        top_right_quadrant_zip_name = 'cropped_top_right_quadrant.zip'

        with zipfile.ZipFile(right_half_zip_name, 'w') as right_zip, \
             zipfile.ZipFile(top_right_quadrant_zip_name, 'w') as top_right_zip:
            for result in tqdm(results, desc="Writing zips"):
                right_zip.writestr(f"right_half_{result['filename']}", result['right_half_bytes'])
                top_right_zip.writestr(f"top_right_{result['filename']}", result['top_right_bytes'])

        print("\n✅ Processing complete!")
        print(f"Created '{right_half_zip_name}' and '{top_right_quadrant_zip_name}'.")

        # --- Display Download Links ---
        print("\n⬇️ Download your new zip files:")
        display(FileLink(right_half_zip_name))
        display(FileLink(top_right_quadrant_zip_name))

    except zipfile.BadZipFile:
        print(f"🚨 ERROR: The file '{path_to_zip}' is not a valid zip file.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# --- Run the main function ---
process_images_in_parallel(zip_file_path)